# Notebook 05 - Logistic Regression and Binary Classification

So far every model we built predicted a number. House price was a number, and our loss measured how far that number landed from the truth. Generally, when we train a model to give us a continuous value, we call it a **regression** problem (and in particular, when we use a linear model, it's called **linear regression**). But a lot of real problems are not about a number at all. Sometimes we want a yes or no answer. Is this email spam or not, does a medical test come back positive or negative... A task where the answer is one of a few fixed categories is called a **classification** task. When there are exactly two categories we call it **binary classification**, and that is the topic of this notebook (otherwise it's called **multiclass classification**).

## A medical dataset with two features

Let us set up a small medical dataset. We have data from a group of patients. For each patient we measured two features, the patient's *blood sugar level* and their *body mass index*, or BMI for short. We also know the result of a diabetes test for each patient. The result is one of two classes. A negative test is class $0$ and a positive test is class $1$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

n0 = 120
sugar0 = rng.normal(100, 15, n0)
bmi0   = rng.normal(26, 4.0, n0)

n1 = 120
sugar1 = rng.normal(132, 18, n1)
bmi1   = rng.normal(32, 5.0, n1)

sugar = np.concatenate([sugar0, sugar1])
bmi   = np.concatenate([bmi0, bmi1])
X = np.column_stack([sugar, bmi])
y = np.concatenate([np.zeros(n0), np.ones(n1)]).astype(int)

C    = ["#9ecae1", "#08519c"]
NAME = ["negative (0)", "positive (1)"]

fig, ax = plt.subplots(figsize=(8, 5.5))
for c in (0, 1):
    m = y == c
    ax.scatter(X[m, 0], X[m, 1], color=C[c], edgecolor="white", s=40,
               label=NAME[c])
ax.set_title("Diabetes dataset, two features and two classes")
ax.set_xlabel("blood sugar level")
ax.set_ylabel("BMI")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

> **Note:** Normally, the above should be a 3D plot, for there are three variables: blood sugar ($x_1$), BMI ($x_2$), and test result ($y$). Instead of drawing a complex 3D graph, we draw a 2D plot with the two features on the axes and use different colors to show the classes $0$ and $1$.

## From a continuous number to a 0 or 1 answer

Our models so far all start the same way. We take the features, multiply each one by its own weight, and add everything up. With our two features $x_1$ and $x_2$, the weighted sum is
$$ z = w_0 + w_1 x_1 + w_2 x_2. $$

Here is the simplest possible idea for classification: fit this line to the patients the way we fit any regression, treating the label as the number to predict, ($0$ for a negative test and a $1$ for a positive one) We minimize the mean squared error between the line and those labels with gradient descent.

Such a model gives us a continuous number for every patient, but we want it to output a class label ($0$ or $1$). So once the line is fit, we turn its output into a class with a **threshold**. We choose a cutoff value $t$, and call every output at or above $t$ class $1$ and everything below it class $0$. This function is called the **step function** and we denote it by $u_t$:
$$ u_t(z) = \begin{cases} 1 & \text{if } z \ge t \\ 0 & \text{if } z < t. \end{cases} $$

Now, let us fit our line to the data as we discussed.

In [ ]:
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression

line = LinearRegression().fit(X, y)

raw_pts = line.predict(X)
print(f"The fitted line outputs values from {raw_pts.min():.2f} to {raw_pts.max():.2f}")

xvals = np.linspace(X[:, 0].min() - 10, X[:, 0].max() + 10, 300)
yvals = np.linspace(X[:, 1].min() - 3,  X[:, 1].max() + 3,  300)
xx, yy = np.meshgrid(xvals, yvals)
grid = np.column_stack([xx.ravel(), yy.ravel()])
raw = line.predict(grid).reshape(xx.shape)

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "xy"}, {"type": "scene"}]],
    column_widths=[0.46, 0.54],
    subplot_titles=("Seen from above: one smooth number",
                    "The same fit is really a tilted plane"),
)

fig.add_contour(
    x=xvals, y=yvals, z=raw, colorscale="Blues",
    contours=dict(showlines=False), opacity=0.7,
    colorbar=dict(title="output of<br>the fitted line", x=0.42, len=0.9),
    row=1, col=1,
)
fig.add_surface(
    x=xvals, y=yvals, z=raw, colorscale="Blues", opacity=0.85, showscale=False,
    row=1, col=2,
)

for c in (0, 1):
    m = y == c
    fig.add_scatter(
        x=X[m, 0], y=X[m, 1], mode="markers", name=NAME[c], legendgroup=NAME[c],
        marker=dict(size=7, color=C[c], line=dict(color="black", width=1)),
        row=1, col=1,
    )
    fig.add_scatter3d(
        x=X[m, 0], y=X[m, 1], z=y[m].astype(float), mode="markers",
        name=NAME[c], legendgroup=NAME[c], showlegend=False,
        marker=dict(size=4, color=C[c], line=dict(color="black", width=1)),
        row=1, col=2,
    )

fig.update_xaxes(title_text="blood sugar level", row=1, col=1)
fig.update_yaxes(title_text="BMI", row=1, col=1)
fig.update_layout(
    title="Fit the line: the same model from above (left) and tilted in 3D (right, drag to rotate)",
    width=1150, height=560,
    scene=dict(
        xaxis_title="blood sugar level",
        yaxis_title="BMI",
        zaxis_title="output / label",
    ),
    legend=dict(x=0.0, y=1.0),
)
fig.show()

> **Note:** Because we have two features (contrary to previous notebooks), what we have fitted is an entire plane instead of a single line. On the right, you'll find a 3D interactive plot of that plane, and on the left, a 2D contour plot of the same plane seen from above, colored by the continuous output of the line.

Now, let us apply the function $u_t$ to the model we just fitted with different values of $t$. The model's labels are determined as:
$$ \hat{y} = u_t(w_0 + w_1 x_1 + w_2 x_2). $$

In [ ]:
ts = [0.25, 0.5, 0.75]

fig, axes = plt.subplots(1, 3, figsize=(15, 5.0), sharex=True, sharey=True)
for ax, t in zip(axes, ts):
    region = (raw >= t).astype(float)             # u_t applied to the fitted line's output
    ax.contourf(xx, yy, region, levels=[-0.5, 0.5, 1.5], cmap="Blues", alpha=0.35)
    ax.contour(xx, yy, raw, levels=[t], colors="black", linewidths=2)
    for c in (0, 1):
        m = y == c
        ax.scatter(X[m, 0], X[m, 1], color=C[c], edgecolor="black", s=22, linewidth=0.5,
                   label=NAME[c])
    acc_t = ((line.predict(X) >= t).astype(int) == y).mean()
    ax.set_title(f"cutoff  t = {t}    (accuracy {acc_t:.0%})")
    ax.set_xlabel("blood sugar level")
axes[0].set_ylabel("BMI")
axes[0].legend(fontsize=8, loc="upper left")
fig.suptitle("Changing the cutoff t slides the decision boundary across the patients")
plt.tight_layout()
plt.show()

Note that every point is forced to one side of the cutoff, separated by the **decision boundary**. As we raise the cutoff from $0.25$ to $0.75$, the decision boundary moves across the plot: a higher $t$ asks for a larger output before the model will call a patient positive, so the paler "negative" region grows and fewer patients come out positive. 

A **misclassification** happens when the model's output is on the wrong side of the cutoff; for example, when a dark blue point (a positive patient) lands in the pale blue region (called negative by the model — **false negative**), or when a pale blue point (a negative patient) lands in the dark blue region (called positive by the model — **false positive**). Accordingly, we can define the model's accuracy as:
$$ \text{accuracy} = \frac{\text{number of correctly classified patients}}{\text{total number of patients}}. $$
In this example, putting the decision boundary at near $t=0.5$ gives us the best accuracy.

## One further step

Often times, we are not satisfied with just a class label, but also want to know how sure the model is about that label. For example, if the model says a patient has a negative test, we want to quantify its confidence in that prediction (e.g., 100% sure vs. 50% sure). Because if the model is only 50% sure, we might want to run more tests before we make a final call.

The function we used, $u_t$, does not give us any of that. It just gives us a hard $0$ or $1$. So we have to go back to the line and ask: How can we read convert the weighted sum of our model $z = w_0 + w_1 x_1 + w_2 x_2$ into a probability? The answer is using the **sigmoid** function, denoted by $\sigma$ (greek letter "sigma"):
$$ \sigma(z) = \frac{1}{1 + e^{-z}}. $$

Let us plot it to see what it does. Below is the sigmoid in red on top of the old step function in gray, so you can see how one is the *smoothed out* version of the other.

In [ ]:
z = np.linspace(-8, 8, 400)
sig  = 1 / (1 + np.exp(-z))
step = (z >= 0).astype(float)

plt.figure(figsize=(7, 4.2))
plt.plot(z, step, color="gray",    lw=1.5, ls="--", label="step function")
plt.plot(z, sig,  color="tab:red", lw=2.5,          label="sigmoid")
plt.axhline(0.5, color="gray", lw=0.8, ls=":")
plt.axvline(0,   color="gray", lw=0.8, ls=":")
plt.xlabel("z  (the weighted sum)")
plt.ylabel("sigma(z)")
plt.legend(fontsize=9)
plt.title("The sigmoid is a smooth version of the step function")
plt.show()

The most important thing to notice is the *range* of the sigmoid. No matter what number we put in, the sigmoid will always give us a number between $0$ and $1$, which works nicely for probabilities.

Read the above shape from left to right. When the weighted sum $z$ is a large negative number the output is almost $0$, and when $z$ is a large positive number the output is almost $1$, exactly like the step function. The difference is what happens in the middle. Instead of one sudden jump, the sigmoid rises gradually, and when $z$ is exactly $0$ the output is exactly $0.5$.

## The logistic regression model

Now we can write the whole model in one line. We take the weighted sum $z = w_0 + w_1 x_1 + w_2 x_2$ and pass it through the sigmoid.
$$ f(x_1, x_2) = \sigma(z) = \frac{1}{1 + e^{-(w_0 + w_1 x_1 + w_2 x_2)}}. $$
We read the output as the probability that the patient has a positive test, in other words the probability of class $1$ (the probability of negative test is then $1 - \text{probability of positive test}$). If the model outputs $0.9$ it is very sure the test is positive. If it outputs $0.2$ it thinks a negative test is much more likely. Underneath it is still the same weighted sum from regression, and the sigmoid only reshapes that number into a probability.

To turn a probability into an actual class we are back to a threshold, $u_t$. The natural cutoff is $t = 0.5$: if the probability sits above $0.5$ we predict class $1$, otherwise class $0$. It is also much more intuitive to choose a cutoff on a fixed probability scale than on an arbitrary number scale.

> **Note:** We should select the cutoff depending on the application. For example, if we're doing cancer diagnosis, we might want to set a very low cutoff, like $t=0.2$, so that we catch as many positive cases as possible, even if it means we get more false positives.

The labels, are again determined according to the cutoff:
$$ \hat{y} = u_t(\sigma(w_0 + w_1 x_1 + w_2 x_2)). $$

## How we train it, the log loss

In the previous example, we used the mean squared error to fit our line. With the sigmoid function, there is a loss function that is much better suited to probabilities: **the log loss**, also known as **binary cross-entropy (BCE)**. For each patient, the model outputs $p$, the predicted probability that the patient belongs to class $1$. If the true label is $1$, we want $p$ to be close to $1$. If the true label is $0$, we want $p$ to be close to $0$. In plain words, the loss punishes the model hard when it is confident and wrong, and barely at all when it is confident and right.

$$
\text{loss} = -\big(y \log p + (1-y)\log(1-p)\big).
$$

If the true label is $1$, only the first part stays and the loss is $-\log p$, which is small when $p$ is close to $1$ and large when $p$ is close to $0$. If the true label is $0$, only the second part stays and the loss is $-\log(1-p)$, which works the same way in reverse.

We train the model the same way as before. We use gradient descent to nudge the weights $w_0, w_1, w_2$ until the total log loss over the patients is as small as we want. To run gradient descent we need the gradient of the loss, so let us derive it cleanly.

For a patient with features $x_1, x_2$ we form the weighted sum and pass it through the sigmoid to get $p$, the predicted probability of class $1$:
$$ z = w_0 + w_1 x_1 + w_2 x_2, \qquad p = \sigma(z) = \frac{1}{1 + e^{-z}}. $$
Across all $n$ patients we average the per-patient log loss, so the quantity we minimize is
$$ L = \frac{1}{n} \sum_{i=1}^{n} \ell_i, \qquad \ell_i = -\big(y_i \log p_i + (1-y_i)\log(1-p_i)\big) \text{ for the } i\text{-th patient}. $$

Remember that what we want is $\partial L / \partial w_j$. Since $L = \frac{1}{n}\sum_i \ell_i$ and the derivative of a sum is the sum of the derivatives, the derivative of $L$ is just the average of the per-patient derivatives:
$$ \frac{\partial L}{\partial w_j} = \frac{1}{n}\sum_{i=1}^{n} \frac{\partial \ell_i}{\partial w_j}. $$
So it suffices to compute $\partial \ell / \partial w_j$ once for any patient, and average it over the patients at the end.

Now look at how one patient's loss depends on a weight $w_j$: the weight feeds the sum $z$, the sum feeds the probability $p$ through the sigmoid, and the probability feeds the loss $\ell$. So $w_j \to z \to p \to \ell$ is a chain of three links, and the chain rule multiplies the three derivatives along it:
$$ \frac{\partial \ell}{\partial w_j} = \frac{\partial \ell}{\partial p}\cdot\frac{\partial p}{\partial z}\cdot\frac{\partial z}{\partial w_j}. $$
(We drop the patient index $i$ while we work on a single patient.) We need each of the three ingredients, so let us derive them one at a time.

**Ingredient 1: $\partial \ell / \partial p$.** Here the label $y$ is a fixed constant (the patient's true $0$ or $1$) and $p$ is the variable, so we differentiate $\ell = -\big(y\log p + (1-y)\log(1-p)\big)$ term by term in $p$. Using $\frac{d}{dp}\log p = \frac{1}{p}$ and $\frac{d}{dp}\log(1-p) = -\frac{1}{1-p}$:
$$ \frac{\partial \ell}{\partial p} = -\left(\frac{y}{p} - \frac{1-y}{1-p}\right). $$

**Ingredient 2: $\partial p / \partial z$ (derivative of the sigmoid).** Writing $\sigma(z) = (1 + e^{-z})^{-1}$:
$$ \sigma'(z) = -\big(1 + e^{-z}\big)^{-2}\cdot\big(-e^{-z}\big) = \frac{e^{-z}}{\big(1 + e^{-z}\big)^2}. $$
Split that into two factors, and rewrite the second by adding and subtracting $1$ in its numerator:
$$ \frac{e^{-z}}{1 + e^{-z}} = \frac{(1 + e^{-z}) - 1}{1 + e^{-z}} = 1 - \frac{1}{1 + e^{-z}} = 1 - \sigma(z). $$
So we can write the derivative of the sigmoid using sigmoid itself:
$$ \frac{\partial p}{\partial z} = \sigma'(z) = \underbrace{\frac{1}{1 + e^{-z}}}_{=\,\sigma(z)}\cdot\underbrace{\frac{e^{-z}}{1 + e^{-z}}}_{=\,1-\sigma(z)} = \sigma(z)\,\big(1 - \sigma(z)\big) = p\,(1-p). $$

**Ingredient 3: $\partial z / \partial w_j$.** The sum $z = w_0 + w_1 x_1 + w_2 x_2$ is linear in the weights, so each derivative is simply the feature that the weight multiplies (and $1$ for the bias $w_0$):
$$ \frac{\partial z}{\partial w_0} = 1, \qquad \frac{\partial z}{\partial w_1} = x_1, \qquad \frac{\partial z}{\partial w_2} = x_2. $$

**Put the ingredients together.** Multiply the three ingredients straight along the chain. The $p(1-p)$ from ingredient 2 cancels the two denominators in ingredient 1, and what survives is the $p - y$, the gap between the predicted probability and the true label:
$$ \frac{\partial \ell}{\partial w_j}
   = \underbrace{-\left(\frac{y}{p} - \frac{1-y}{1-p}\right)}_{\partial \ell/\partial p}
     \cdot \underbrace{p(1-p)}_{\partial p/\partial z}
     \cdot \frac{\partial z}{\partial w_j}
   = \big(\underbrace{-y(1-p) + (1-y)p}_{=\,p-y}\big)\,\frac{\partial z}{\partial w_j}
   = (p - y)\,\frac{\partial z}{\partial w_j}. $$
Substituting the three values of $\partial z / \partial w_j$ gives the per-patient partials:
$$ \frac{\partial \ell}{\partial w_0} = (p - y), \qquad \frac{\partial \ell}{\partial w_1} = (p - y)\,x_1, \qquad \frac{\partial \ell}{\partial w_2} = (p - y)\,x_2. $$

**The gradient.** Now plug these per-patient partial derivatives back into the average $\frac{\partial L}{\partial w_j} = \frac{1}{n}\sum_i \frac{\partial \ell_i}{\partial w_j}$ from the start, which gives the three components of the gradient of $L$:
$$ \frac{\partial L}{\partial w_0} = \frac{1}{n}\sum_{i=1}^{n} (p_i - y_i), \qquad
   \frac{\partial L}{\partial w_1} = \frac{1}{n}\sum_{i=1}^{n} (p_i - y_i)\,x_{i1}, \qquad
   \frac{\partial L}{\partial w_2} = \frac{1}{n}\sum_{i=1}^{n} (p_i - y_i)\,x_{i2}. $$

($x_{i1}$ and $x_{i2}$ are the two features of the $i$-th patient.)
Remember that the gradient is a vector of all three partial derivatives, so we can write it as
$$ \nabla L = \begin{bmatrix} \frac{\partial L}{\partial w_0} \\ \frac{\partial L}{\partial w_1} \\ \frac{\partial L}{\partial w_2} \end{bmatrix} = \frac{1}{n}\sum_{i=1}^{n} (p_i - y_i)\,\begin{bmatrix} 1 \\ x_{i1} \\ x_{i2} \end{bmatrix}. $$

Gradient descent takes the usual step with learning rate $\eta$:
$$ \mathbf{w} \leftarrow \mathbf{w} - \eta\, \nabla L. $$

Let us actually train the model and plot it:

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X, y)

w1, w2 = model.coef_[0]
w0 = model.intercept_[0]
print(f"learned weights   w0 = {w0:.3f},  w1 = {w1:.3f},  w2 = {w2:.3f}")

xx, yy = np.meshgrid(
    np.linspace(X[:, 0].min() - 10, X[:, 0].max() + 10, 300),
    np.linspace(X[:, 1].min() - 3,  X[:, 1].max() + 3,  300),
)
grid_pts = np.column_stack([xx.ravel(), yy.ravel()])
prob = model.predict_proba(grid_pts)[:, 1].reshape(xx.shape)

fig, ax = plt.subplots(figsize=(8, 5.5))
cf = ax.contourf(xx, yy, prob, levels=20, cmap="Blues", alpha=0.85)
plt.colorbar(cf, ax=ax, label="probability of a positive test")
for c in (0, 1):
    m = y == c
    ax.scatter(X[m, 0], X[m, 1], color=C[c], edgecolor="white", s=40,
               label=NAME[c])
ax.set_xlabel("blood sugar level")
ax.set_ylabel("BMI")
ax.legend(fontsize=8, loc="upper left")
ax.set_title("The model assigns a smooth probability across the plane")
plt.show()

## More complex models, curvier boundaries

Logistic regression always draws a single straight decision boundary. That is part of its appeal, it is simple and steady, but it is also a limit. Think back to the regression notebooks, where we moved beyond a straight line by giving the model more flexibility, for example with higher degree polynomials, so it could bend and wiggle to follow the data far more closely. The same idea carries over to classification. More flexible models are free to draw curved and complex boundaries instead of one straight cut.

The figure below sets our logistic regression beside one such flexible model:

In [ ]:
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

complex_model = make_pipeline(
    StandardScaler(),
    SVC(kernel="rbf", gamma=8, C=1000, probability=True, random_state=0),
).fit(X, y)

prob_complex = complex_model.predict_proba(grid_pts)[:, 1].reshape(xx.shape)

fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 5.4), sharex=True, sharey=True)
panels = [
    (axL, prob,         model,         "Simple: logistic regression"),
    (axR, prob_complex, complex_model, "Complex: flexible model"),
]
for ax, P, mdl, title in panels:
    region = (P >= 0.5).astype(float)
    ax.contourf(xx, yy, region, levels=[-0.5, 0.5, 1.5], cmap="Blues", alpha=0.35)
    ax.contour(xx, yy, P, levels=[0.5], colors="black", linewidths=2)
    for c in (0, 1):
        m = y == c
        ax.scatter(X[m, 0], X[m, 1], color=C[c], edgecolor="black", s=28, linewidth=0.5,
                   label=NAME[c])
    acc_m = (mdl.predict(X) == y).mean()
    ax.set_title(f"{title}   (accuracy {acc_m:.1%})")
    ax.set_xlabel("blood sugar level")
axL.set_ylabel("BMI")
axL.legend(fontsize=8, loc="upper left")
fig.suptitle("A straight boundary versus a complex, wiggly one")
plt.tight_layout()
plt.show()

The wiggly boundary is bending itself around individual patients, including the ones sitting in the overlap zone where the two classes genuinely mix. This is the overfitting trap we met with complex regression models back in Notebook 03: a boundary flexible enough to wrap around every training point is *memorizing* training examples rather than learning the real pattern, and it can do worse on new patients than the plain straight line.

## Summary

- Classification predicts a category instead of a number. With two categories we call it binary classification, and we label the classes $0$ and $1$.
- Logistic regression takes the weighted sum of features and passes it through the sigmoid function, which squashes the result into a probability between $0$ and $1$.
- We train with the log loss, which punishes confident mistakes hard, and we minimize it with gradient descent just as before.